In [22]:
import os
import torch
import numpy as np
import pandas as pd
import json
from tqdm import tqdm
from PIL import Image
import pickle
from torchvision import models, transforms

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import f1_score, top_k_accuracy_score

import numpy as np
import pandas as pd
import os

In [8]:
df = pd.read_csv("../data/processed/spectrograms/manifest.csv")

output_dir = "../data/processed/transferlearningData"
os.makedirs(output_dir, exist_ok=True)

# manter apenas arquivos válidos
df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

print("Total de amostras:", len(df))
df.head()

Total de amostras: 30166


,image_path,label,audioSource,roi_start,roi_end,roi_min_freq,roi_max_freq,roi_duration
0,../data/processed/spectrograms\images\Megascop...,Megascops choliba,W04856768S2013814_20230811_033000.WAV,38.071627,41.774503,0.601953,1.266653,3.702877
1,../data/processed/spectrograms\images\Leptotil...,Leptotila verreauxi,W04856768S2013814_20230811_072000.WAV,6.038564,7.289066,0.373524,0.554806,1.250502
2,../data/processed/spectrograms\images\Leptotil...,Leptotila verreauxi,W04856768S2013814_20230811_072000.WAV,9.049033,10.067961,0.334678,0.541857,1.018928
3,../data/processed/spectrograms\images\Leptotil...,Leptotila verreauxi,W04856768S2013814_20230811_072000.WAV,19.099367,20.396185,0.347627,0.541857,1.296817
4,../data/processed/spectrograms\images\Leptotil...,Leptotila verreauxi,W04856768S2013814_20230811_072000.WAV,24.332952,25.815029,0.321729,0.554806,1.482077


In [9]:
# =========================
# DEVICE E TRANSFORM
# =========================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# CNN → Global Average Pooling → vetor fixo
transform = transforms.Compose([
    transforms.ToTensor(),          # Cada imagem vira um tensor - (3, H, W)
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # Média e desvio padrão da ImageNet
        std=[0.229, 0.224, 0.225],
    ),
])


In [10]:
# =========================
# INICIALIZAÇÃO DOS MODELOS 
# =========================

def build_model(model_name):

    if model_name == "resnet":
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)

        # remove classificador final
        model.fc = torch.nn.Identity()

        feature_dim = 2048

    elif model_name == "efficientnet":
        weights = models.EfficientNet_B2_Weights.DEFAULT
        model = models.efficientnet_b2(weights=weights)

        # remove classificador final
        model.classifier = torch.nn.Identity()

        feature_dim = 1408

    else:
        raise ValueError("Modelo inválido")

    # Move o modelo para GPU (CUDA) ou CPU
    model = model.to(DEVICE)
    # Coloca a rede em modo de inferência
    model.eval()               

    return model, feature_dim

In [11]:
# =========================
# EXTRAÇÃO DE FEATURES
# =========================

def extract_features(df, model):

    features = []

    with torch.no_grad():

        for _, row in tqdm(df.iterrows(), total=len(df)):

            img = Image.open(row["image_path"]).convert("RGB")
            x = transform(img)
            x = x.unsqueeze(0).to(DEVICE)

            f = model(x)
            f = f.cpu().numpy().flatten() #flatten: (1, C) → (C,)

            extra = np.array([
                row["roi_min_freq"],
                row["roi_max_freq"],
                row["roi_duration"],
            ], dtype=np.float32)

            f_final = np.concatenate([f, extra])

            features.append(f_final)

    return np.array(features)

### Extração Features ResNet

In [12]:
model_resnet, dim_resnet = build_model("resnet")

# Extrai features
X_resnet = extract_features(df, model_resnet)

print("Shape ResNet:", X_resnet.shape)
print("NaNs:", np.isnan(X_resnet).sum())
print("Infs:", np.isinf(X_resnet).sum())

# Normalização
scaler_resnet = StandardScaler()
X_resnet_scaled = scaler_resnet.fit_transform(X_resnet)

print("Shape ResNet Scaled:", X_resnet_scaled.shape)
print(X_resnet_scaled.nbytes / 1e6, "MB")

np.save(os.path.join(output_dir, "X_resnet_scaled.npy"), X_resnet_scaled)

100%|██████████| 30166/30166 [21:07<00:00, 23.80it/s] 


Shape ResNet: (30166, 2051)
NaNs: 0
Infs: 0
Shape ResNet Scaled: (30166, 2051)
247.481864 MB


### Extração Features EfficientNet B2

In [13]:
model_eff, dim_eff = build_model("efficientnet")

# Extrai features
X_eff = extract_features(df, model_eff)

print("Shape EfficientNet:", X_eff.shape)
print("NaNs:", np.isnan(X_eff).sum())
print("Infs:", np.isinf(X_eff).sum())

# Normalização
scaler_eff = StandardScaler()
X_eff_scaled = scaler_eff.fit_transform(X_eff)

print("Shape EfficientNet Scaled:", X_eff_scaled.shape)
print(X_eff_scaled.nbytes / 1e6, "MB")

np.save(os.path.join(output_dir, "X_efficientnet_scaled.npy"), X_eff_scaled)

100%|██████████| 30166/30166 [14:42<00:00, 34.19it/s] 


Shape EfficientNet: (30166, 1411)
NaNs: 0
Infs: 0
Shape EfficientNet Scaled: (30166, 1411)
170.256904 MB


### Salvando Labels e LabelEncoder

In [14]:
encoder = LabelEncoder()
y = encoder.fit_transform(df["label"])

np.save(os.path.join(output_dir, "y.npy"), y)

with open(os.path.join(output_dir, "label_encoder.pkl"), "wb") as f:
    pickle.dump(encoder, f)

In [15]:
K_FOLDS = 5
TOP_K = 5

Cs = [1, 10, 100]
gammas = ['scale', 'auto', 1e-2, 1e-3]

RESULTS_DIR = "./results_svm"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [18]:
# Features
X_resnet = np.load("../data/processed/transferlearningData/X_resnet_scaled.npy")
X_eff = np.load("../data/processed/transferlearningData/X_efficientnet_scaled.npy")

# Manifest
df = pd.read_csv("../data/processed/spectrograms/manifest.csv")

# Labels (já codificados!)
y = np.load("../data/processed/transferlearningData/y.npy")

# Groups (ESSENCIAL)
groups = df["audioSource"].values

print("Shapes:")
print("ResNet:", X_resnet.shape)
print("EfficientNet:", X_eff.shape)
print("y:", y.shape)

Shapes:
ResNet: (30166, 2051)
EfficientNet: (30166, 1411)
y: (30166,)


In [19]:
def selecionar_melhor_svm(X_train, X_val, y_train, y_val):

    best_f1 = -1
    best_params = None

    for C in Cs:
        for gamma in gammas:
            svm = SVC(C=C, gamma=gamma, probability=True)
            svm.fit(X_train, y_train)

            pred = svm.predict(X_val)
            f1 = f1_score(y_val, pred, average="macro")

            if f1 > best_f1:
                best_f1 = f1
                best_params = (C, gamma)

    # treino final com treino + val
    X_full = np.vstack([X_train, X_val])
    y_full = np.concatenate([y_train, y_val])

    final_model = SVC(C=best_params[0], gamma=best_params[1], probability=True)
    final_model.fit(X_full, y_full)

    return final_model, best_params, best_f1

In [24]:
def rodar_experimento(nome_modelo, X, y, groups):

    print(f"\n===== RODANDO: {nome_modelo} =====")

    sgkf = StratifiedGroupKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

    resultados = []

    for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):

        print(f"\n--- Fold {fold+1} ---")

        X_train_full = X[train_idx]
        y_train_full = y[train_idx]

        X_test = X[test_idx]
        y_test = y[test_idx]

        # 🔥 CONVERTER PRA PANDAS (igual seu script)
        y_train_full_series = pd.Series(y_train_full)

        # 🔹 VERIFICAÇÃO DE CLASSES
        counts = y_train_full_series.value_counts()

        if counts.min() >= 2:
            print("Split estratificado")
            X_train, X_val, y_train, y_val = train_test_split(
                X_train_full,
                y_train_full,
                stratify=y_train_full,
                test_size=0.2,
                random_state=42
            )
        else:
            print("⚠️ Classe com 1 amostra → split SEM estratificação")
            X_train, X_val, y_train, y_val = train_test_split(
                X_train_full,
                y_train_full,
                test_size=0.2,
                random_state=42
            )

        # 🔹 SCALER (igual seu script)
        scaler = StandardScaler()
        scaler.fit(X_train)

        X_train = scaler.transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)

        # 🔹 GRID SEARCH
        svm, best_params, val_f1 = selecionar_melhor_svm(
            X_train, X_val, y_train, y_val
        )

        # 🔹 TESTE
        y_pred = svm.predict(X_test)
        y_proba = svm.predict_proba(X_test)

        f1 = f1_score(y_test, y_pred, average="macro")

        # 🔥 MESMO TRATAMENTO DO SEU SCRIPT
        y_test_series = pd.Series(y_test)

        desconhecidas = (~y_test_series.isin(svm.classes_)).sum()
        print(f"Amostras com classe não vista: {desconhecidas}/{len(y_test)}")

        mask = y_test_series.isin(svm.classes_)
        y_test_filtrado = y_test_series[mask]
        y_proba_filtrado = y_proba[mask.values]

        # 🔥 AJUSTE DE CLASSES (igual seu script)
        classes_presentes = np.intersect1d(svm.classes_, np.unique(y_test_filtrado))
        idxs = [np.where(svm.classes_ == c)[0][0] for c in classes_presentes]

        y_proba_filtrado = y_proba_filtrado[:, idxs]

        topk = top_k_accuracy_score(
            y_test_filtrado,
            y_proba_filtrado,
            k=TOP_K,
            labels=classes_presentes
        )

        print(f"F1: {f1:.4f}")
        print(f"Top-{TOP_K}: {topk:.4f}")

        resultados.append({
            "fold": fold,
            "f1": float(f1),
            "topk": float(topk),
            "best_C": best_params[0],
            "best_gamma": str(best_params[1]),
            "val_f1": float(val_f1),
            "unknown_samples": int(desconhecidas)
        })

    return resultados

In [ ]:
result_resnet = rodar_experimento("ResNet", X_resnet, y, groups)
result_eff = rodar_experimento("EfficientNet", X_eff, y, groups)


===== RODANDO: ResNet =====


c:\Users\Pichau\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:994: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(



--- Fold 1 ---
⚠️ Classe com 1 amostra → split SEM estratificação


In [ ]:
X_concat = np.concatenate([X_resnet, X_eff], axis=1)

result_concat = rodar_experimento("ResNet+EfficientNet", X_concat, y, groups)

In [ ]:
def salvar_resultados(nome, resultados):

    path = os.path.join(RESULTS_DIR, f"{nome}.json")

    with open(path, "w") as f:
        json.dump(resultados, f, indent=4)

    f1s = [r["f1"] for r in resultados]
    topks = [r["topk"] for r in resultados]

    resumo = {
        "f1_mean": float(np.mean(f1s)),
        "f1_std": float(np.std(f1s)),
        "topk_mean": float(np.mean(topks)),
        "topk_std": float(np.std(topks)),
    }

    with open(os.path.join(RESULTS_DIR, f"{nome}_resumo.json"), "w") as f:
        json.dump(resumo, f, indent=4)

    print(f"\nResumo {nome}:")
    print(resumo)

In [ ]:
salvar_resultados("resnet", result_resnet)
salvar_resultados("efficientnet", result_eff)
salvar_resultados("ensemble", result_concat)